---

### <center> 전처리 이후 최종 결합

---

In [24]:
import pandas as pd
import glob
import os
import warnings
import re

# 경고 문구 무시
warnings.filterwarnings('ignore')

#### <center> 1. 구글 트렌드 지역 가중 연산


In [25]:
# 구글 트렌드 지역 가중치 연산
trends_files = glob.glob("../Data Folder/구글 트렌드/google_*.csv")
weight_data = []

for f in trends_files:
    try:
        # 구글 트렌드는 상단 2줄이 메타데이터라 skiprows=2
        df = pd.read_csv(f, skiprows=2)
        if '지역' in df.columns:
            # 헤더에서 카테고리명만 추출 (예: '가디건: (24...)' -> '가디건')
            cat_col = [c for c in df.columns if ':' in c][0]
            category = cat_col.split(':')[0].strip()
                
            for _, row in df.iterrows():
                region = row['지역']
                score = row[cat_col]
                # 결측치나 '<1' 같은 문자열 처리
                if pd.isna(score): score = 0
                elif isinstance(score, str) and '<' in score: score = 0.5
                    
                weight_data.append({
                    'standard_category': category, 
                    'region_name': region, 
                    'weight_score': float(score)
                })
    except Exception as e:
        print(f"⚠️ {f} 처리 중 오류: {e}")

df_weights = pd.DataFrame(weight_data)
    
# 가중치 정규화 (카테고리별 지역 총합이 1.0이 되도록)
df_weights['normalized_weight'] = df_weights.groupby('standard_category')['weight_score'].transform(lambda x: x / x.sum() if x.sum() != 0 else 0)

# 기상청/네이버와 매핑할 2글자 지역명 변환 딕셔너리
region_map = {
    '서울특별시': '서울', '부산광역시': '부산', '대구광역시': '대구', '인천광역시': '인천',
    '광주광역시': '광주', '대전광역시': '대전', '울산광역시': '울산', '경기도': '경기',
    '강원도': '강원', '충청북도': '충북', '충청남도': '충남', '전라북도': '전북',
    '전라남도': '전남', '경상북도': '경북', '경상남도': '경남', '제주특별자치도': '제주'
}
df_weights['mapped_region'] = df_weights['region_name'].map(region_map)

#### <center> 2. 플랫폼별 상품 리뷰 데이터 로드

In [26]:
# 파일 로드 (경로 확인 필수)
path_29 = r"../Data Folder/29cm dataset/product_29cm.csv"
path_zig_prod = r"../Data Folder/Zigzag dataset/product_zigzag.csv"
path_rev_29 = r"../Data Folder/29cm dataset/review_29cm.csv"
path_rev_zig = r"../Data Folder/Zigzag dataset/review_zigzag_total.csv"

# 상품 데이터 통합
df_prod_29 = pd.read_csv(path_29) if os.path.exists(path_29) else pd.read_csv("product_29cm.csv")
df_prod_zig = pd.read_csv(path_zig_prod)
df_product = pd.concat([df_prod_29, df_prod_zig], ignore_index=True)
df_product['product_id'] = df_product['product_id'].astype(str)

# 리뷰 데이터 통합
df_rev_29 = pd.read_csv(path_rev_29) if os.path.exists(path_rev_29) else pd.read_csv("review_29cm.csv")
df_rev_zig = pd.read_csv(path_rev_zig)
df_review = pd.concat([df_rev_29, df_rev_zig], ignore_index=True)
df_review['product_id'] = df_review['product_id'].astype(str)


#### <center> 3. 리뷰 날짜 데이터 정규화 및 전처리

In [27]:
# [핵심 전처리] 하이브리드 날짜 변환 로직
def robust_date_parser(date_series):
    # 1. 숫자 형태(타임스탬프)인 것들만 골라내기
    is_numeric = date_series.astype(str).str.isnumeric()
        
    # 2. 숫자형은 seconds 단위 타임스탬프로 변환
    converted_numeric = pd.to_datetime(pd.to_numeric(date_series[is_numeric], errors='coerce'), unit='s')
        
    # 3. 이미 날짜 형식인 문자열들 변환
    converted_string = pd.to_datetime(date_series[~is_numeric], errors='coerce')
        
    # 4. 두 결과를 합치고 포맷 통일 (YYYY-MM-DD)
    return converted_numeric.combine_first(converted_string).dt.strftime('%Y-%m-%d')

df_review['date'] = robust_date_parser(df_review['review_date'])
    
# 날짜가 제대로 안 바뀐 데이터는 과감히 제거 (데이터 무결성)
df_review = df_review.dropna(subset=['date'])

#### <center> 4. 상품 리뷰 결합 및 전국 일별 판매량 집계

In [28]:
# 상품 + 리뷰 결합
df_sales = pd.merge(df_review, df_product[['product_id', 'standard_category']], on='product_id', how='inner')

# 전국구 일자별/카테고리별 판매량 집계
df_daily_sales = df_sales.groupby(['date', 'standard_category']).size().reset_index(name='national_sales_volume')

#### <center> 5. 네이버 검색 트렌드 데이터 결합

In [29]:
# 네이버 검색 트렌드 데이터 로드
search_path = r"../Data Folder\네이버 데이터랩\search_trend_raw.csv"
df_search = pd.read_csv(search_path)
df_search['date'] = pd.to_datetime(df_search['date']).dt.strftime('%Y-%m-%d')
df_search.rename(columns={'search_volume': 'national_search_volume'}, inplace=True)
    
# 판매량과 검색량 통합 (Outer Join을 통해 데이터 보존)
df_demand = pd.merge(df_daily_sales, df_search[['date', 'standard_category', 'national_search_volume']], 
                         on=['date', 'standard_category'], how='outer')
df_demand = df_demand.fillna(0)

#### <center> 6. 가중치를 이용한 전국 데이터를 지역별 데이터로 분배

In [30]:
# 전국구 데이터에 16개 지역 가중치를 곱해 행을 늘림
df_regional_demand = pd.merge(df_demand, df_weights[['standard_category', 'mapped_region', 'normalized_weight']], 
                               on='standard_category', how='inner')
    
# 지역별 분배량 계산
df_regional_demand['regional_sales_volume'] = (df_regional_demand['national_sales_volume'] * df_regional_demand['normalized_weight'])
df_regional_demand['regional_search_volume'] = (df_regional_demand['national_search_volume'] * df_regional_demand['normalized_weight'])

print("🗺️ 지역별 수요 분배 완료")

🗺️ 지역별 수요 분배 완료


#### <center> 7. 기상 데이터 결합 및 최종 마스터 테이블 생성

In [31]:
weather_path = r"../Data Folder\기상청 종관\weather_feature.csv"

if os.path.exists(weather_path):
    df_weather = pd.read_csv(weather_path)
    df_weather['date'] = pd.to_datetime(df_weather['date']).dt.strftime('%Y-%m-%d')
        
    # 기상청 데이터의 지역 컬럼명 자동 탐색
    weather_region_col = [c for c in df_weather.columns if 'region' in c.lower()][0]
        
    # [날짜, 지역] 기준으로 결합
    df_final = pd.merge(df_weather, df_regional_demand, 
                        left_on=['date', weather_region_col], 
                        right_on=['date', 'mapped_region'], 
                        how='inner')
                            
    # 컬럼 정리
    cols_to_keep = ['date', 'mapped_region', 'standard_category'] + \
                    [c for c in df_weather.columns if c not in ['date', weather_region_col]] + \
                    ['regional_search_volume', 'regional_sales_volume']
    
    df_final = df_final[cols_to_keep]
    df_final.rename(columns={'mapped_region': 'region'}, inplace=True)
        
    # 최종 마스터 테이블 저장 (필요 시 주석 해제)
    # df_final.to_csv("final_mart.csv", index=False, encoding='utf-8-sig')
    
    print("\n" + "="*65)
    print(f"✨ 마스터 데이터(final_mart.csv) 생성 완료")
    print(f"    - 총 데이터 건수: {len(df_final)}행")
    print("="*65)
    
    # 샘플 데이터 확인
    display(df_final.head())
else:
    print("weather_feature.csv 파일이 없어 병합을 완료할 수 없습니다.")


✨ 마스터 데이터(final_mart.csv) 생성 완료
    - 총 데이터 건수: 33180행


,date,region,standard_category,avg_temp,diurnal_range,precipitation,regional_search_volume,regional_sales_volume
0,2024-01-01,서울,가디건,3.3,7.6,0.0,0.232240,0.189394
1,2024-01-01,서울,바람막이,3.3,7.6,0.0,0.323977,0.000000
2,2024-01-01,서울,봄 자켓,3.3,7.6,0.0,0.929104,0.075640
3,2024-01-01,서울,블레이저,3.3,7.6,0.0,0.244063,0.569569
4,2024-01-01,서울,야상,3.3,7.6,0.0,0.153046,0.073529
